# TCGA KIRC — Data Exploration

**Project:** Kidney Renal Clear Cell Carcinoma (KIRC) Survival Analysis  
**Purpose:** Raw data inspection, structure verification, and patient ID alignment across all three source datasets.

This notebook consolidates two early exploratory steps:
- **Phase 1** — Load data files, inspect shapes, columns, and sample values
- **Phase 2** — Align patient IDs across clinical, follow-up, and expression datasets; verify cohort overlap

---

## Phase 1: Data Load & Structure Check

We load the three TCGA-KIRC source files and inspect their shape, columns, and values.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"

clinical   = pd.read_csv(DATA_DIR / "clinical.tsv",        sep="\t")
follow_up  = pd.read_csv(DATA_DIR / "follow_up.tsv",       sep="\t")
expression = pd.read_csv(DATA_DIR / "kirc_expression.tsv", sep="\t")

print("Files loaded successfully!")
print(f"  clinical shape:    {clinical.shape}")
print(f"  follow_up shape:   {follow_up.shape}")
print(f"  expression shape:  {expression.shape}")

### 1.1 First Rows

In [ ]:
print("=== Clinical Data (first 5 rows) ===")
clinical.head()

In [ ]:
print("=== Follow-Up Data (first 5 rows) ===")
follow_up.head()

In [ ]:
print("=== Expression Data (first 5 rows, first 8 columns) ===")
expression.iloc[:5, :8]

### 1.2 Column Inspection

In [ ]:
print("=== Clinical Column Names ===")
print(list(clinical.columns))

print("\n=== Follow-Up Column Names ===")
print(list(follow_up.columns))

print("\n=== Expression Data — First 10 Columns ===")
print(list(expression.columns[:10]))
print(f"Total expression columns: {len(expression.columns)}")

### 1.3 Identify Likely Patient ID Columns

We search for columns containing common TCGA ID keywords to find the correct join key.

In [ ]:
id_keywords = ["id", "case", "patient", "submitter", "barcode"]

print("=== Clinical columns that may contain patient IDs ===")
for col in clinical.columns:
    if any(kw in col.lower() for kw in id_keywords):
        print(f"  {col}")

print("\n=== Follow-Up columns that may contain patient IDs ===")
for col in follow_up.columns:
    if any(kw in col.lower() for kw in id_keywords):
        print(f"  {col}")

In [ ]:
# Show sample values from ID columns
print("=== Clinical — sample ID values (first 5) ===")
for col in clinical.columns:
    if any(kw in col.lower() for kw in ["submitter_id", "case_id", "patient"]):
        print(f"  {col}: {clinical[col].head().tolist()}")

print("\n=== Follow-Up — sample ID values (first 5) ===")
for col in follow_up.columns:
    if any(kw in col.lower() for kw in ["submitter_id", "case_id", "patient"]):
        print(f"  {col}: {follow_up[col].head().tolist()}")

---
## Phase 2: Patient ID Alignment

TCGA expression data uses **sample barcodes** (e.g. `TCGA-BP-4162-01`) while clinical/follow-up use **patient barcodes** (`TCGA-BP-4162`). We need to normalise the expression IDs and verify the overlap across all three datasets.

In [ ]:
# Separate gene ID column from TCGA sample columns
non_tcga  = [c for c in expression.columns if not c.startswith("TCGA")]
tcga_cols = [c for c in expression.columns if c.startswith("TCGA")]

print(f"Non-TCGA columns (gene identifiers): {non_tcga}")
print(f"TCGA sample columns:                 {len(tcga_cols)}")
print(f"First 5 barcodes: {tcga_cols[:5]}")

In [ ]:
# Extract patient IDs from expression barcodes
# TCGA-XX-XXXX-01  →  TCGA-XX-XXXX  (drop sample-type suffix)
expr_patient_ids = list(set("-".join(c.split("-")[:3]) for c in tcga_cols))

print(f"Unique patient IDs in expression data: {len(expr_patient_ids)}")
print(f"Example IDs: {sorted(expr_patient_ids)[:5]}")

In [ ]:
# Count unique patients per dataset
# NOTE: use cases.submitter_id (TCGA barcode), NOT cases.case_id (UUID — won't match expression)
clin_id_col = "cases.submitter_id"
fu_id_col   = "cases.submitter_id"

n_clin = clinical[clin_id_col].nunique()
n_fu   = follow_up[fu_id_col].nunique()
n_expr = len(expr_patient_ids)

print("Unique patients per dataset:")
print(f"  Clinical   ({clin_id_col}): {n_clin}")
print(f"  Follow-up  ({fu_id_col}):   {n_fu}")
print(f"  Expression (from barcodes): {n_expr}")

In [ ]:
# Cross-dataset overlap check
clin_set = set(clinical[clin_id_col])
fu_set   = set(follow_up[fu_id_col])
expr_set = set(expr_patient_ids)

print(f"Patients in clinical ∩ follow-up:    {len(clin_set & fu_set)}")
print(f"Patients in clinical ∩ expression:   {len(clin_set & expr_set)}")
print(f"Patients in follow-up ∩ expression:  {len(fu_set & expr_set)}")
print(f"Patients in ALL three datasets:      {len(clin_set & fu_set & expr_set)}")
print()
print("→ Final cohort available after inner merge across all three: ",
      len(clin_set & fu_set & expr_set), "patients")

---
## Summary

| Dataset | Unique Patients | Notes |
|---------|----------------|-------|
| Clinical | ~537 | TCGA nested TSV, `cases.submitter_id` is the join key |
| Follow-up | ~537 | Supplement for censored patients missing follow-up time |
| Expression | ~534 | RNA-seq, barcodes trimmed to patient ID (`TCGA-XX-XXXX`) |
| **Inner overlap** | **~533** | **Final cohort used in the analysis pipeline** |

**Key finding:** `cases.submitter_id` (TCGA barcode format `TCGA-XX-XXXX`) is the correct join key — `cases.case_id` is a UUID that does not appear in expression barcodes.

The full analysis pipeline — survival target construction, feature engineering, model training and evaluation — is in `03_survival_analysis_report.ipynb`.